# Across-session pupil position in the eye-anchored frame

**Question.** Some days the online gaze looks centered, others biased. Real gaze difference, or an
artifact of the online processing?

Pupil position *in the image* isn't comparable across days (each day is a different **crop**/zoom of a
fixed camera view; the head-fixed animal may sit slightly differently). So we express the pupil in an
**eye-anchored frame** from clicked landmarks: `u` runs corner-to-corner (−1 .. +1, 0 = centered),
invariant to crop/translation/zoom. We also run the **online tracker** (`get_pupil_online`) beside a
**robust ellipse** detector — agreement means the online tracker is faithful.

**Read-out** (`centered` vs `biased`): `u` differs → **real gaze bias**; `u` same though image
position differed → **crop/head/processing artifact**. Helpers in `eyevideo.py`.


In [ ]:
import eyevideo as ev
ev.ANIMAL_DIR = "/mnt/at-storageB1_I/EyeVideo/AT-B1NO1"
N = 200                                 # frames sampled per session (used everywhere below)
print(len(ev.list_sessions()), "sessions available")


## 1. Click eye landmarks for each date (only if missing)
Do one date at a time: set `d`, run the widget cell (clicks only if that date has no landmarks),
left-click ≥5 points, right-click to undo, then `clk.save()`. Repeat for every date.
No-click fallback: `ev.show_mean_grid(d)` then `ev.set_landmarks(d, [(x,y), ...])`.


In [ ]:
%matplotlib widget
d = "2025-08-15"                       # <- change per date
clk = ev.clicker_if_missing(d)         # clicks only if this date has no landmarks yet


In [ ]:
clk and clk.save()                     # run after clicking; repeat the two cells for the next date
%matplotlib inline
print("landmarks on file:", sorted(ev.LANDMARKS))


## 2. Label your days into two conditions, then track once
Tracking is computed once per session at `N` frames and cached (in memory + on disk in `.track_cache/`), so every plot below reuses the stored values — even after a kernel restart.


In [ ]:
biased_dates   = ['2026-06-04', '2026-05-20', '2026-05-12', '2026-04-17']
centered_dates = ['2026-05-27', '2026-04-30', '2026-06-11', '2026-06-09']

tracked = ev.track_dates(centered_dates + biased_dates, n=N)   # track once, store, reuse


## 3. QC: tracking per date
Ellipse should hug the pupil; markers coincide; agreement scatter on the identity line.


In [ ]:
ev.show_tracking_examples(sorted(ev.LANDMARKS), k=5)


In [ ]:
ev.tracker_agreement(sorted(ev.LANDMARKS), n=N)


## 4. Does the tracker discrepancy differ between conditions?
Ellipse-x vs online-x and ellipse-y vs online-y split by condition, plus `delta = online - ellipse`
per group. If `delta` is similar across conditions, a tracking artifact can't explain a group
difference; if it differs, it could contribute.


In [ ]:
_ = ev.compare_agreement(centered_dates, biased_dates, n=N)


## 5. Compare conditions: pupil position in the eye frame
Density histograms of eye-frame position per condition, for both trackers and **both axes**
(`u` = horizontal / eye-frame x, `v` = vertical / eye-frame y). The test asks whether the **mean**
eye-frame position differs between conditions, run on one summary per session (`summary='mean'`;
independent unit = session, n = #days): two groups → Welch t-test, >2 → one-way ANOVA. A
frame-level test is also printed but is pseudo-replicated and only illustrative.
Needs ≥2 days/condition for the valid test. For >2 groups: `conditions={'name': [dates], ...}`.


In [ ]:
res = ev.compare_conditions(centered_dates, biased_dates, n=N, bins=60)
